# 05. LLM Decoding 전략 실습

## 실습 개요
본 실습에서는 `transformers` 기반으로 sLLM(소형 LLM)인 `google/gemma-2-2b-it` 모델을 로드하고, 입력 전처리(plain prompt vs Chat Template)부터 생성 결과 분석(토큰별 score 확인)까지 **추론 파이프라인 전체 흐름**을 실습합니다. 또한 `generate()`의 주요 디코딩 파라미터를 바꿔가며 **출력 다양성/일관성의 트레이드오프**를 비교합니다.

## 실습 목표
- `AutoTokenizer`, `AutoModelForCausalLM`로 sLLM 모델/토크나이저를 로딩하고 추론 준비를 할 수 있다.
- 텍스트 입력을 토크나이징하고(`tokenizer(...)`) Chat Format 입력을 구성할 수 있다(`tokenizer.apply_chat_template`).
- `model.generate()` 결과를 디코딩하고(`batch_decode`) 입력/출력 경계를 올바르게 분리해 확인할 수 있다.
- `return_dict_in_generate=True`, `output_scores=True`를 사용해 **생성 단계별 logits/score**를 확인하고, 간단한 argmax 기반 해석을 수행할 수 있다.
- 다양한 디코딩 전략을 적용하고 결과를 비교해 이해한다.
  - **Deterministic(유사 Greedy) 생성**
  - **Sampling 기반 생성**
  - **확률 질량 제한 샘플링**

## Prerequisites
- **Python**: 3.9+ 권장
- **Library**: `torch`, `transformers`, `huggingface_hub`
- **Access**: Hugging Face 토큰(`huggingface_hub.login`) 및 모델 접근 권한
- **Hardware**: GPU 권장



In [1]:
! pip install torch transformers -U

In [2]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os

/home/xeong_oxx/miniconda3/envs/qlora_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it", device_map="auto")

model.eval()
model.generation_config.max_new_tokens = 1024

Loading weights: 100%|██████████| 288/288 [00:00<00:00, 428.70it/s, Materializing param=model.norm.weight]                                


In [4]:
input_text='오늘 저녁 뭐먹지??'

input_ids= tokenizer(input_text,return_tensors='pt').to('cuda')

In [5]:
input_text='오늘 저녁 뭐먹지??'

In [6]:
input_ids

{'input_ids': tensor([[     2, 237410, 240703,  80404, 245029, 235248, 245365, 241877, 236183,
           4481]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [7]:
outputs=model.generate(**input_ids)

RuntimeError: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cuda:6 (when checking argument in method wrapper_CUDA__index_select)

In [ ]:
outputs

tensor([[     2, 237410, 240703,  80404, 245029, 235248, 245365, 241877, 236183,
           4481, 235248, 240800,    109, 238721, 245029, 130886, 236417, 236214,
         235248, 249942, 222449, 236392, 236183,  46749, 238754, 235832,  99805,
         237214, 236569, 244572, 235336, 235248,    109,    688, 238918, 235248,
         249942, 227991, 204551, 235336,    688, 235248,    109, 235287,   5231,
         236464, 236386,  66058, 235248, 246264,    139, 243960, 236183, 236464,
         236386, 235269,  44997, 236464, 236386, 235269, 235248, 245049, 236464,
         236386,  73143,    108, 235287,   5231, 237138, 238325, 238256,  66058,
         235248, 247217,    139, 237410, 241330, 236770, 235269,  50833, 236770,
         235269, 120816, 237924,  73143,    108, 235287,   5231, 240478, 237433,
          66058, 235248, 246887,    139, 238780, 238312, 239043, 235269, 129669,
         236375, 242019, 236432, 235269,  84815, 239969,  73143,    108, 235287,
           5231, 238264, 238

In [ ]:
print(tokenizer.batch_decode(outputs,skip_special_tokens=True)[0][len(input_text):].lstrip())

🤤

저녁 식사는 뭘 먹을지 고민이 많으시죠? 

**🤔 뭘 좋아하세요?** 

* **고기:** 🥩  돼지고기, 소고기, 닭고기 등
* **해산물:** 🦐  오징어, 연어, 참치 등
* **채소:** 🥦  양배추, 브로콜리, 당근 등
* **간식:** 🍪  과자, 빵, 초콜릿 등
* **한식:** 🍜  비빔밥, 김치찌개, 떡볶이 등
* **양식:** 🍕  피자, 샌드위치, 햄버거 등
* **중식:** 🍣  라멘, 돈까스, 떡볶이 등
* **일식:** 🍤  구이, 볶음, 샐러드 등

**✨ 혹시 특별한 날인가요?** 

* **날짜:** 📅  (예: 토요일, 일요일, 특별한 날)
* **사람:** 👫  (예: 친구, 가족, 연인)

**💡 팁:** 

* **저녁 식사 시간:** ⏰  (예: 7시, 8시, 9시)
* **식사 스타일:** 🍽️  (예: 혼밥, 식사, 외식)
* **예산:** 💰  (예: 저렴, 중간, 고급)

**💖 궁금한 점이 있으면 언제든지 물어보세요!** 

**😊 즐거운 저녁 식사 되세요!** 



In [ ]:
chat_text = tokenizer.apply_chat_template( [
    {'role':'user','content':input_text}],tokenize=False, add_generation_prompt=True
                                           )

In [ ]:
print(chat_text)

<bos><start_of_turn>user
오늘 저녁 뭐먹지??<end_of_turn>
<start_of_turn>model



In [ ]:
input_ids = tokenizer(chat_text, return_tensors='pt').to('cuda')

In [ ]:
input_ids.input_ids

tensor([[     2,      2,    106,   1645,    108, 237410, 240703,  80404, 245029,
         235248, 245365, 241877, 236183,   4481,    107,    108,    106,   2516,
            108]], device='cuda:0')

In [ ]:
outputs= model.generate(
    input_ids=input_ids.input_ids,
    return_dict_in_generate=True,
    output_scores=True
)

In [ ]:
outputs

GenerateDecoderOnlyOutput(sequences=tensor([[     2,      2,    106,   1645,    108, 237410, 240703,  80404, 245029,
         235248, 245365, 241877, 236183,   4481,    107,    108,    106,   2516,
            108, 237410, 240703,  80404, 245029, 235248, 245365, 222449, 236392,
         236183,  46749, 238754, 235832, 236569, 239265, 237526, 235341,  44416,
            139, 236770, 242251, 107342, 238186, 236392, 227991, 204551, 235336,
         235248,    109, 243976,  70231,  34103, 237533, 239055,  78183, 238994,
         237014, 236569, 237722, 235248, 246506, 207221, 236214,  75943, 239250,
         236392,  56787, 237135, 241949,  22618, 215995, 237526, 235341, 235248,
            109, 235287,   5231, 236770, 242251,  86126, 239758, 236137, 107342,
         238186, 236392, 227991, 204551, 235336,    688,    591, 238748, 235292,
          35191, 238186, 235269,  47250, 238186, 235269,  32929, 238186, 235269,
         119452, 238186, 235269,  70754, 238186,  73143, 235275,    108, 

In [ ]:
len(tokenizer)

256000

In [ ]:
logits = outputs.scores

In [ ]:
len(logits)

228

In [ ]:
len(logits[0][0])

256000

In [ ]:
one_logits=logits[14][0]

In [ ]:
one_logits

tensor([-18.8085,  -7.2078,  -8.3807,  ..., -18.0425, -14.7094, -18.1198],
       device='cuda:0')

In [ ]:
predict_id=torch.argmax(one_logits).item()

In [ ]:
predict_id

237526

In [ ]:
tokenizer.decode([predict_id])

'요'

In [ ]:
len(outputs.sequences[0])

247

In [ ]:
len(input_ids.input_ids[0])

19

In [ ]:
print(tokenizer.batch_decode(outputs.sequences,skip_special_tokens=True)[0])

user
오늘 저녁 뭐먹지??
model
오늘 저녁 뭐 먹을지 고민이시군요! 😊  어떤 음식을 좋아하세요? 

좀 더 자세히 알려주시면 딱 맞는 추천을 해드릴 수 있어요! 

* **어떤 종류의 음식을 좋아하세요?** (예: 한식, 중식, 일식, 양식, 분식 등)
* **매운 음식을 좋아하세요?** 🌶️
* **매운 음식은 괜찮으신가요?** 🌶️
* **어떤 분위기의 식사를 원하세요?** (예: 편안한, 멋진, 깔끔한, 흥미로운 등)
* **가격대는 어느 정도 생각하세요?** 💰
* **집에서 혹은 외식을 원하세요?** 🏠 

조금 더 알려주시면 딱 맞는 맛있는 저녁 식사를 추천해드릴게요! 😉 



In [ ]:
text='세종대왕이 어느나라 사람이니?'

chat_text= tokenizer.apply_chat_template( [
    {'role':'user','content':text}],tokenize=False, add_generation_prompt=True
                                           )


# 둘의 차이도 확인!
input_ids=tokenizer(input_text,return_tensors='pt').to('cuda')

# input_ids =. tokenizer(chat_text,return_tensors='pt').to('cuda')

In [ ]:
outputs=model.generate(input_ids=input_ids.input_ids,
                       return_dict_in_generate=True,
                       output_scores=True,
                       temperature=0.0,
                      #  top_k=10,
                      #  top_p= 0.9,
                      #  do_sample=True
                       )

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
print(tokenizer.batch_decode(outputs.sequences,skip_special_tokens=True)[0])

user
세종대왕이 어느나라 사람이니?
model
세종대왕은 **조선**의 왕이었습니다. 

세종대왕은 조선의 14대 왕으로, 1443년부터 1450년까지 재위했습니다. 



In [ ]:
outputs=model.generate(input_ids=input_ids.input_ids,
                       return_dict_in_generate=True,
                       output_scores=True,
                       temperature=2.0,
                      #  top_k=10,
                      #  top_p= 0.9,
                       min_p=0.2,
                       do_sample=True
                       )

In [ ]:
print(tokenizer.batch_decode(outputs.sequences,skip_special_tokens=True)[0])

user
세종대왕이 어느나라 사람이니?
model
세종대왕은 **대한민국** 사람입니다. 😄  그는 조선의 왕으로, 1443년부터 1450년까지 재위했습니다. 



In [ ]:
from transformers import pipeline


In [ ]:
pipe=pipeline(model=model, task='text-generation', tokenizer=tokenizer)

Device set to use cuda:0


In [ ]:
# 아래도 챗 템플릿 형태로 바꿔보세요!
input_text='대한민국의 수도는 어디인가요?'

In [ ]:
result=pipe(input_text)

In [ ]:
print(result[0]['generated_text'][len(input_text):].lstrip())

정답: **서울** 입니다. 



###**콘텐츠 라이선스**
<font color='red'><b>**(주)업스테이지가 제공하는 모든 교육 콘텐츠의 지식재산권은
운영 주체인 (주)업스테이지 또는 해당 저작물의 적법한 관리자에게 귀속되어 있습니다.**</b></font>

콘텐츠 일부 또는 전부를 **복사, 복제, 판매, 재판매 공개, 공유** 등을 할 수 없습니다. 유출될 경우 지식재산권 침해에 대한 책임을 부담할 수 있습니다.

유출에 해당하여 금지되는 행위의 예시는 다음과 같습니다.
* 콘텐츠를 재가공하여 온/오프라인으로 공개하는 행위
* 콘텐츠의 일부 또는 전부를 이용하여 인쇄물을 만드는 행위
* 콘텐츠의 전부 또는 일부를 녹취 또는 녹화하거나 녹취록을 작성하는 행위
* 콘텐츠의 전부 또는 일부를 스크린 캡쳐하거나 카메라로 촬영하는 행위
* 지인을 포함한 제3자에게 콘텐츠의 일부 또는 전부를 공유하는 행위
* 다른 정보와 결합하여 Upstage Education의 콘텐츠임을 알아볼 수 있는 저작물을 작성, 공개하는 행위
* 제공된 데이터의 일부 혹은 전부를 Upstage Education 프로젝트/실습 수행 이외의 목적으로 사용하는 행위